# T32 金融资负 重构

本项目用于获取和处理金融机构资产负债表数据。

## 项目概述

- **任务ID**: T32
- **功能**: 金融机构资产负债数据获取、清洗和存储
- **数据源**: Wind EDB接口、同花顺EDB接口
- **应用场景**: 银行业研究、金融机构分析、宏观经济研究

---

## 1. 项目概述

### 1.1 功能说明

本模块实现以下核心功能：

1. **金融资负数据获取**：从Wind和同花顺接口获取金融机构资产负债表数据
2. **基金净申购数据**：每日更新基金净申购数据
3. **数据清洗处理**：清理重复数据，确保数据质量
4. **数据存储管理**：将处理后的数据存储到MySQL数据库

### 1.2 数据更新策略

- **金融资负数据**：每月20-30号更新上月数据
- **基金净申购数据**：每天更新

---

## 2. 环境配置

### 2.1 导入依赖包

In [ ]:
# 标准库
import os
import sys
import time
from datetime import datetime, timedelta

# 第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text, inspect
import pymysql

# 数据源接口（需要安装相应客户端）
# from WindPy import w
# from iFinDPy import *

# 导入配置
from config import Config

print(f"环境配置完成 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

### 2.2 初始化数据库连接

In [ ]:
# 创建数据库引擎
def create_db_engine():
    """创建数据库连接引擎"""
    db_config = Config.get_database_config()
    connection_string = (
        f"mysql+pymysql://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_config['database']}"
    )
    engine = sqlalchemy.create_engine(
        connection_string,
        pool_recycle=3600,
        pool_pre_ping=True
    )
    return engine

sql_engine = create_db_engine()
print("数据库连接创建成功")

### 2.3 初始化数据源接口

In [ ]:
def init_data_sources():
    """初始化Wind和同花顺数据源"""
    try:
        # 初始化Wind
        # w.start()
        print("Wind接口初始化... (需要安装WindPy)")
    except Exception as e:
        print(f"Wind初始化失败: {e}")
    
    try:
        # 初始化同花顺
        # THS_iFinDLogin(Config.THS_USER, Config.THS_PASSWORD)
        print("同花顺接口初始化... (需要安装iFinDPy)")
    except Exception as e:
        print(f"同花顺初始化失败: {e}")

# init_data_sources()

---

## 3. 数据获取

### 3.1 数据代码配置

定义Wind和同花顺的数据代码列表。

In [ ]:
# Wind数据代码列表
WIND_CODES = [
    "M0001538", "M0001527", "M0251904", "M0001528", "M0001529", "M0001530",
    "M0062047", "M0062845", "M0062846", "M0251905", "M0001533", "M0001534",
    "M0251906", "M0251907", "M0062848", "M0001536", "M0001537", "M0001557",
    "M0001539", "M0061954", "M0001540", "M0251908", "M0001542", "M0001545",
    "M0251909", "M0001543", "M0001547", "M0001541", "M0001544", "M0001548",
    "M0001549", "M0001550", "M0001551", "M0251910", "M0251911", "M0001552",
    "M0251912", "M0061955", "M0001554", "M0150191", "M0062849", "M0001556",
    "M0251956", "M0251940", "M0251941", "M0251942", "M0251943", "M0251944",
    "M0251945", "M0251946", "M0251947", "M0251948", "M0251949", "M0251950",
    "M0251951", "M0251952", "M0251953", "M0251954", "M0251955", "M0251977",
    "M0251957", "M0251958", "M0251959", "M0251960", "M0251961", "M0251962",
    "M0251963", "M0251964", "M0251965", "M0333070", "M0333071", "M0251966",
    "M0333072", "M0251967", "M0251968", "M0251969", "M0251970", "M0251971",
    "M0251972", "M0251973", "M0333073", "M0251974", "M0251975", "M0251976",
    "M0048455", "M0048441", "M0252060", "M0062879", "M0048445", "M0252061",
    "M0252062", "M0062881", "M0062876", "M0048442", "M0048443", "M0048444",
    "M0062878", "M0252063", "M0252064", "M0252065", "M0252066", "M0048451",
    "M0252068", "M0048452", "M0048453", "M0048454", "M0048471", "M0048456",
    "M0061993", "M0048457", "M0048466", "M0061994", "M0061995", "M0061996",
    "M0048468", "M0150196", "M0062883", "M0252069", "M0048469", "M0048470",
    "M0009940", "M0043410", "M0043412", "M0043411", "M0043413", "M0009947",
    "M0009969", "M0043417", "M0043418", "M0009978", "M0043419", "M0001380",
    "M0001382", "M0001384", "M0001386", "M0010131", "M0001485", "M0001486",
    "M0001487", "M0001488", "M0001489", "M0001490", "M0001491", "M0001492",
    "M0001494", "M0001493", "M0001495", "M0001496", "M0001497", "M0001504",
    "M0001498", "M0001499", "M0001500", "M5639029", "M5639030", "M5639031",
    "M5639032", "M5639033", "M5639034", "M5639035", "J3426133", "M0010125",
    "M5525755", "M5525756", "M5525757", "M5525758", "M5525759", "M5525760",
    "M5525761", "M5525762", "M6179494", "M6094230", "M6094231", "Y7375557",
    "M0001705", "M0001724", "M5449834", "M5524595", "M5207551", "M0001680",
    "M0001684", "M0001685", "M0001686", "M0001687", "M0001688", "M0001689",
    "M0001690", "M0068054", "M0001697", "M0001698", "M0001699", "M0001700",
    "M0001701", "M0001702"
]

# 同花顺数据代码列表
THS_CODES = [
    'S004345997', 'S004346069', 'S004346029', 'S004345944', 'S004346101'
]

# 基金代码列表
FUND_CODES = ['511090.SH', '511130.SH']

print(f"Wind数据代码数量: {len(WIND_CODES)}")
print(f"同花顺数据代码数量: {len(THS_CODES)}")
print(f"基金代码数量: {len(FUND_CODES)}")

### 3.2 时间工具函数

In [ ]:
def get_last_month_end():
    """获取上个月的最后一天日期"""
    today = datetime.now()
    first_day_this_month = today.replace(day=1)
    last_month_end = first_day_this_month - timedelta(days=1)
    return last_month_end

def is_update_period():
    """检查是否在每月20-30号的更新期间"""
    today = datetime.now()
    return 20 <= today.day <= 30

# 测试
last_month = get_last_month_end()
print(f"上个月最后一天: {last_month.strftime('%Y-%m-%d')}")
print(f"当前是否在更新期间(20-30号): {is_update_period()}")

### 3.3 Wind数据获取函数

In [ ]:
def fetch_wind_data(code, start_date, end_date):
    """
    从Wind EDB接口获取数据
    
    参数:
        code: 数据代码
        start_date: 开始日期 (YYYY-MM-DD)
        end_date: 结束日期 (YYYY-MM-DD)
    
    返回:
        DataFrame: 包含日期和值的数据
    """
    try:
        # 实际使用时取消注释
        # error_code, wsd_data = w.edb(code, start_date, end_date, usedf=True)
        # if error_code == 0 and not wsd_data.empty:
        #     return wsd_data
        print(f"[模拟] 获取Wind数据: {code}, {start_date} 至 {end_date}")
        return pd.DataFrame()
    except Exception as e:
        print(f"获取Wind数据失败: {code}, 错误: {e}")
        return None

### 3.4 同花顺数据获取函数

In [ ]:
def fetch_ths_data(code, start_date, end_date):
    """
    从同花顺EDB接口获取数据
    
    参数:
        code: 数据代码
        start_date: 开始日期 (YYYY-MM-DD)
        end_date: 结束日期 (YYYY-MM-DD)
    
    返回:
        DataFrame: 包含时间(time)和值(value)的数据
    """
    try:
        # 实际使用时取消注释
        # df = THS_EDB(code, '', start_date, end_date)
        # if df.data is not None:
        #     return df.data
        print(f"[模拟] 获取同花顺数据: {code}, {start_date} 至 {end_date}")
        return pd.DataFrame()
    except Exception as e:
        print(f"获取同花顺数据失败: {code}, 错误: {e}")
        return None

### 3.5 基金净申购数据获取

In [ ]:
def fetch_fund_net_inflow(code, date):
    """
    获取基金净申购数据
    
    参数:
        code: 基金代码
        date: 日期 (YYYY-MM-DD)
    
    返回:
        float: 净申购金额
    """
    try:
        # 实际使用时取消注释
        # wsd_data = w.wsd(code, "mf_netinflow", date, date, "unit=1")
        # if wsd_data.ErrorCode == 0 and wsd_data.Data[0]:
        #     return wsd_data.Data[0][0]
        print(f"[模拟] 获取基金净申购: {code}, 日期: {date}")
        return None
    except Exception as e:
        print(f"获取基金净申购失败: {code}, 错误: {e}")
        return None

---

## 4. 数据处理

### 4.1 数据格式标准化

In [ ]:
def process_wind_data(wsd_data, code):
    """
    处理Wind返回的数据
    
    参数:
        wsd_data: Wind返回的原始数据
        code: 数据代码
    
    返回:
        DataFrame: 处理后的数据
    """
    if wsd_data is None or wsd_data.empty:
        return None
    
    try:
        # 处理单值数据
        if wsd_data.index[0] == code:
            df = pd.DataFrame({
                'dt': [datetime.now()],
                code: [wsd_data['CLOSE'].iloc[0]]
            })
        else:
            # 处理时间序列数据
            wsd_data.index = pd.to_datetime(wsd_data.index)
            df = wsd_data[['CLOSE']].copy()
            df.columns = [code]
            df = df.reset_index().rename(columns={'index': 'dt'})
        
        # 只保留每个月最新的数据
        df['year_month'] = df['dt'].dt.strftime('%Y-%m')
        df = df.sort_values('dt').groupby('year_month').last().reset_index()
        df = df.drop('year_month', axis=1)
        
        return df
    except Exception as e:
        print(f"处理Wind数据失败: {e}")
        return None


def process_ths_data(ths_data, code):
    """
    处理同花顺返回的数据
    
    参数:
        ths_data: 同花顺返回的原始数据
        code: 数据代码
    
    返回:
        DataFrame: 处理后的数据
    """
    if ths_data is None or ths_data.empty:
        return None
    
    try:
        if 'time' not in ths_data.columns or 'value' not in ths_data.columns:
            print("同花顺返回数据格式不正确")
            return None
        
        df = ths_data[['time', 'value']].copy()
        df.columns = ['dt', code]
        df['dt'] = pd.to_datetime(df['dt'])
        
        # 只保留每个月最新的数据
        df['year_month'] = df['dt'].dt.strftime('%Y-%m')
        df = df.sort_values('dt').groupby('year_month').last().reset_index()
        df = df.drop('year_month', axis=1)
        
        return df
    except Exception as e:
        print(f"处理同花顺数据失败: {e}")
        return None

### 4.2 重复数据检测

In [ ]:
def check_duplicate_data(table_name='金融资负'):
    """
    检查表中是否存在重复数据
    
    参数:
        table_name: 表名
    
    返回:
        dict: 重复数据统计信息
    """
    stats = {
        'total_records': 0,
        'duplicate_months': 0,
        'records_to_delete': 0
    }
    
    try:
        with sql_engine.begin() as connection:
            # 总记录数
            result = pd.read_sql(text(f'SELECT COUNT(*) as total FROM {table_name}'), connection)
            stats['total_records'] = int(result.iloc[0]['total'])
            
            # 存在重复的月份数
            duplicate_query = f'''
            SELECT COUNT(*) as duplicate_months
            FROM (
                SELECT YEAR(dt) as year, MONTH(dt) as month
                FROM {table_name}
                GROUP BY YEAR(dt), MONTH(dt)
                HAVING COUNT(*) > 1
            ) t
            '''
            result = pd.read_sql(text(duplicate_query), connection)
            stats['duplicate_months'] = int(result.iloc[0]['duplicate_months'])
            
            # 需要删除的记录数
            delete_query = f'''
            SELECT SUM(cnt - 1) as delete_count
            FROM (
                SELECT YEAR(dt) as year, MONTH(dt) as month, COUNT(*) as cnt
                FROM {table_name}
                GROUP BY YEAR(dt), MONTH(dt)
                HAVING COUNT(*) > 1
            ) t
            '''
            result = pd.read_sql(text(delete_query), connection)
            if result.iloc[0]['delete_count'] is not None:
                stats['records_to_delete'] = int(result.iloc[0]['delete_count'])
    
    except Exception as e:
        print(f"检查重复数据失败: {e}")
    
    return stats

---

## 5. 核心逻辑

### 5.1 更新数据库记录

In [ ]:
def update_database(code, date, value, table_name='金融资负'):
    """
    更新数据到数据库
    
    参数:
        code: 数据代码/列名
        date: 日期
        value: 数据值
        table_name: 表名
    
    返回:
        bool: 是否成功
    """
    try:
        with sql_engine.begin() as connection:
            # 检查列是否存在
            inspector = inspect(sql_engine)
            columns = [col['name'] for col in inspector.get_columns(table_name)]
            
            if code not in columns:
                # 添加新列
                connection.execute(text(f"ALTER TABLE {table_name} ADD COLUMN {code} FLOAT"))
                print(f"  已添加新列: {code}")
            
            # 检查日期记录是否存在
            check_query = f"SELECT COUNT(*) as count FROM {table_name} WHERE dt = :dt"
            result = pd.read_sql(text(check_query), connection, params={'dt': date})
            
            if result.iloc[0]['count'] > 0:
                # 更新现有记录
                update_query = f"UPDATE {table_name} SET {code} = :value WHERE dt = :dt"
                connection.execute(text(update_query), {'value': value, 'dt': date})
            else:
                # 插入新记录
                insert_query = f"INSERT INTO {table_name} (dt, {code}) VALUES (:dt, :value)"
                connection.execute(text(insert_query), {'dt': date, 'value': value})
            
            print(f"  数据更新成功: {code} = {value}, 日期: {date}")
            return True
            
    except Exception as e:
        print(f"  更新数据库失败: {e}")
        return False

### 5.2 批量更新金融资负数据

In [ ]:
def update_financial_data_batch(codes, data_type, target_date):
    """
    批量更新金融资负数据
    
    参数:
        codes: 数据代码列表
        data_type: 1=Wind, 2=同花顺
        target_date: 目标日期
    
    返回:
        int: 成功更新的数量
    """
    success_count = 0
    target_date_str = target_date.strftime('%Y-%m-%d')
    
    # 计算查询起始日期（3个月前）
    start_date = (target_date - timedelta(days=90)).strftime('%Y-%m-%d')
    
    for code in codes:
        print(f"正在获取数据: {code}")
        
        if data_type == 1:
            # Wind接口
            data = fetch_wind_data(code, start_date, target_date_str)
            if data is not None and not data.empty:
                processed = process_wind_data(data, code)
                if processed is not None and not processed.empty:
                    latest = processed.iloc[-1]
                    if update_database(code, latest['dt'], latest[code]):
                        success_count += 1
        else:
            # 同花顺接口
            data = fetch_ths_data(code, start_date, target_date_str)
            if data is not None and not data.empty:
                processed = process_ths_data(data, code)
                if processed is not None and not processed.empty:
                    latest = processed.iloc[-1]
                    if update_database(code, latest['dt'], latest[code]):
                        success_count += 1
        
        # 添加延迟避免请求过于频繁
        time.sleep(0.1)
    
    return success_count

### 5.3 清理重复数据

In [ ]:
def clean_duplicate_data(table_name='金融资负', dry_run=True):
    """
    清理重复数据，只保留每月最后一天的数据
    
    参数:
        table_name: 表名
        dry_run: 是否只检查不执行
    
    返回:
        bool: 是否成功
    """
    # 先检查重复数据
    stats = check_duplicate_data(table_name)
    
    print(f"总记录数: {stats['total_records']}")
    print(f"存在重复的月份: {stats['duplicate_months']}")
    print(f"需要删除的记录数: {stats['records_to_delete']}")
    
    if stats['records_to_delete'] == 0:
        print("没有需要清理的重复数据")
        return True
    
    if dry_run:
        print("[DRY RUN] 将删除重复记录，只保留每月最后一天的数据")
        return False
    
    try:
        with sql_engine.begin() as connection:
            # 创建临时表存储应保留的日期
            connection.execute(text('DROP TEMPORARY TABLE IF EXISTS temp_keep_dates'))
            create_temp = f'''
            CREATE TEMPORARY TABLE temp_keep_dates AS (
                SELECT MAX(dt) as keep_dt
                FROM {table_name}
                GROUP BY YEAR(dt), MONTH(dt)
            )
            '''
            connection.execute(text(create_temp))
            
            # 删除不应保留的记录
            delete_query = f'''
            DELETE FROM {table_name}
            WHERE dt NOT IN (SELECT keep_dt FROM temp_keep_dates)
            '''
            result = connection.execute(text(delete_query))
            deleted_rows = result.rowcount
            
            # 清理临时表
            connection.execute(text('DROP TEMPORARY TABLE IF EXISTS temp_keep_dates'))
            
            print(f"成功删除 {deleted_rows} 条重复记录")
            return True
            
    except Exception as e:
        print(f"清理重复数据失败: {e}")
        return False

---

## 6. 执行与测试

### 6.1 主执行函数

In [ ]:
def main(dry_run=True):
    """
    主执行函数
    
    参数:
        dry_run: 是否只检查不执行
    """
    print("=" * 50)
    print("金融资负数据取数程序")
    print(f"执行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"模式: {'DRY RUN' if dry_run else 'EXECUTE'}")
    print("=" * 50)
    
    # 1. 检查重复数据
    print("\n1. 检查重复数据...")
    clean_duplicate_data(dry_run=True)
    
    # 2. 检查是否需要更新金融资负数据
    print("\n2. 检查是否需要更新金融资负数据...")
    if is_update_period():
        target_date = get_last_month_end()
        print(f"当前在更新期间（每月20-30号）")
        print(f"目标更新日期: {target_date.strftime('%Y-%m-%d')}")
        
        if not dry_run:
            # 更新Wind数据
            print("\n更新Wind数据...")
            wind_success = update_financial_data_batch(WIND_CODES, 1, target_date)
            print(f"Wind数据更新完成: {wind_success}/{len(WIND_CODES)}")
            
            # 更新同花顺数据
            print("\n更新同花顺数据...")
            ths_success = update_financial_data_batch(THS_CODES, 2, target_date)
            print(f"同花顺数据更新完成: {ths_success}/{len(THS_CODES)}")
        else:
            print("[DRY RUN] 跳过数据更新")
    else:
        print("当前不在更新期间（每月20-30号），跳过金融资负数据更新")
    
    print("\n" + "=" * 50)
    print("程序执行完成")
    print("=" * 50)

### 6.2 执行主程序

In [ ]:
# 执行主程序（DRY RUN模式）
main(dry_run=True)

---

## 7. 工具函数（可复用）

In [ ]:
def get_table_info(table_name='金融资负'):
    """
    获取表的基本信息
    
    参数:
        table_name: 表名
    
    返回:
        dict: 表信息
    """
    info = {'columns': [], 'row_count': 0, 'date_range': {}}
    
    try:
        with sql_engine.begin() as connection:
            # 获取列信息
            inspector = inspect(sql_engine)
            columns = inspector.get_columns(table_name)
            info['columns'] = [col['name'] for col in columns]
            
            # 获取行数
            result = pd.read_sql(text(f'SELECT COUNT(*) as count FROM {table_name}'), connection)
            info['row_count'] = int(result.iloc[0]['count'])
            
            # 获取日期范围
            if info['row_count'] > 0:
                result = pd.read_sql(
                    text(f'SELECT MIN(dt) as min_dt, MAX(dt) as max_dt FROM {table_name}'),
                    connection
                )
                info['date_range'] = {
                    'start': str(result.iloc[0]['min_dt']),
                    'end': str(result.iloc[0]['max_dt'])
                }
    except Exception as e:
        print(f"获取表信息失败: {e}")
    
    return info


def export_data_to_csv(table_name='金融资负', output_path=None):
    """
    导出表数据到CSV文件
    
    参数:
        table_name: 表名
        output_path: 输出路径
    
    返回:
        str: 输出文件路径
    """
    if output_path is None:
        output_path = f'./output/{table_name}_{datetime.now().strftime("%Y%m%d")}.csv'
    
    try:
        df = pd.read_sql(text(f'SELECT * FROM {table_name} ORDER BY dt'), sql_engine)
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"数据已导出到: {output_path}")
        return output_path
    except Exception as e:
        print(f"导出数据失败: {e}")
        return None


def get_latest_values(table_name='金融资负', codes=None):
    """
    获取最新数据值
    
    参数:
        table_name: 表名
        codes: 数据代码列表，None表示所有列
    
    返回:
        dict: 最新数据值
    """
    try:
        query = f'SELECT * FROM {table_name} ORDER BY dt DESC LIMIT 1'
        df = pd.read_sql(text(query), sql_engine)
        
        if df.empty:
            return {}
        
        result = df.iloc[0].to_dict()
        
        if codes:
            result = {k: v for k, v in result.items() if k in codes or k == 'dt'}
        
        return result
    except Exception as e:
        print(f"获取最新数据失败: {e}")
        return {}

In [ ]:
# 测试工具函数
print("测试工具函数:")
print("\n1. 获取表信息:")
# table_info = get_table_info()
# print(f"列数: {len(table_info['columns'])}")
# print(f"行数: {table_info['row_count']}")
print("[需要数据库连接]")

print("\n2. 获取最新数据:")
# latest = get_latest_values()
# print(latest)
print("[需要数据库连接]")

---

## 附录：参考资料

### 数据源说明

1. **Wind EDB接口**
   - 提供宏观经济数据、金融数据等
   - 需要安装WindPy客户端
   - 数据代码格式：M开头（如M0001538）

2. **同花顺EDB接口**
   - 提供经济数据
   - 需要安装iFinDPy
   - 数据代码格式：S开头（如S004345997）

### 数据库表结构

- **金融资负**表：存储金融机构资产负债数据
  - dt: 日期（主键）
  - 各指标列（FLOAT类型）

- **基金净申购**表：存储基金净申购数据
  - dt: 日期
  - trade_code: 基金代码
  - 净申购: 金额

---

*生成时间: 2026-02-15*